In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
# Cấu hình Confluent Kafka
KAFKA_SERVER = "pkc-ldvr1.asia-southeast1.gcp.confluent.cloud:9092"
KAFKA_API_KEY = "LXF7KCAF7HH4LG4C"
KAFKA_API_SECRET = "cfltN30DvSA1L9BIEYAhkI/HelSSGX96AwRFG5npmt9Ef4Ihkv/uEYOcbQbQOZtw"
KAFKA_TOPIC = "binance-kline-1m"

In [0]:
trades_volume_path = "/Volumes/workspace/bronze/binance_raw/trades"
trades_checkpoint = "/Volumes/workspace/bronze/binance_raw/_checkpoints/trades"

## LOAD TRADES DATA

In [0]:
df_trades_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .load(trades_volume_path)
)

# Add metadata
df_bronze_trades = df_trades_stream \
    .withColumnRenamed("value", "raw_payload") \
    .withColumn("ingested_at", current_timestamp())

# Write to trades Table
query_trades = (df_bronze_trades.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", trades_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.bronze.trades") 
)

query_trades.awaitTermination()


In [0]:
%sql
-- SELECT * FROM workspace.bronze.trades LIMIT 10;